<a href="https://colab.research.google.com/github/malokasheko/nlp_assignments/blob/main/assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — From N-Grams to Logistic Regression


## Setup

In [2]:
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# A. loading dataset and Pre-processing, Language Models, and Perplexity

### A1. Task contract

- **Input:** one IMDB movie review (raw text, may contain HTML)
- **Output:** 1 for positive sentiment and 0 for negative sentiment
- **Data:** balanced subset of IMDB (Hugging Face): 5,000 train (2,500 / 2,500) and 1,000 test (500 / 500)
- **Generative LM classifier:** train one bigram LM per class; label a review with the class whose LM gives the **lower perplexity**
- **Metrics:** accuracy and F1-score (positive class), compared with a majority-label baseline
- **Rules:** only standard Python and NumPy for cleaning, features, and modeling. The Hugging Face `datasets` library is used only to download the raw data.

### A2. Load the raw dataset

We use the Hugging Face `datasets` library **only** to download the raw IMDB data . It comes with an official `train` and `test` split (25k reviews each, labels `0` = negative, `1` = positive).

In [3]:
from datasets import load_dataset

try:
    imdb = load_dataset("imdb")
except Exception:
    # Fallback to the fully qualified namespace/name identifier if 'imdb' fails to resolve
    imdb = load_dataset("stanfordnlp/imdb")

print(imdb)

raw_train_examples = [(ex["text"], int(ex["label"])) for ex in imdb["train"]]
raw_test_examples = [(ex["text"], int(ex["label"])) for ex in imdb["test"]]
raw_examples = raw_train_examples + raw_test_examples

assert all(label in {0, 1} for _, label in raw_examples), "unexpected label value"
print("Raw train reviews:", len(raw_train_examples), dict(Counter(l for _, l in raw_train_examples)))
print("Raw test reviews: ", len(raw_test_examples), dict(Counter(l for _, l in raw_test_examples)))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Raw train reviews: 25000 {0: 12500, 1: 12500}
Raw test reviews:  25000 {0: 12500, 1: 12500}


### A3. Inspect raw examples before choosing cleaning steps

In [4]:
for text, label in random.Random(0).sample(raw_examples, 3):
    print(f"[label={label}] {text[:350]}...\n")



[label=0] I'm trying to find something of value here. The best I can muster is that Truffaut wanted to make a movie as tedious, painful, puerile, annoying, illogical, and brainless as the experience of being in love. If that was his goal, then he succeeded, but the solution to his exercise is really a drag to watch.<br /><br />There is one scene that screams...

[label=1] Screening as part of a series of funny shorts at the Sydney Gay and Lesbian Mardi Gras film festival, this film was definitely a highlight. The script is great and the direction and acting was terrific. As another posting said, the actors' comedic timing really made this film. Lots of fun....

[label=0] This movie was, unfortunately, terrible.<br /><br />Clichéd, hackneyed, stilted dialogue and acting make it almost unwatchable.<br /><br />The feel-good finale is laughably lame.<br /><br />There is a reason Judge Reinhold's career has vanished.<br /><br />If you don't live in New York, and aren't Jewish, several of th

**Findings → cleaning decisions**

1. HTML line breaks (`<br />`) are markup, not language → replace tags with a space.
2. Mixed casing would split `Good` / `good` → lowercase.
3. Runs like `...` or `?!` would create empty "sentences" → collapse each run to its first character.
4. Curly apostrophes → straight apostrophe so `don’t` and `don't` match.
5. Duplicate reviews exist → drop them before splitting, so no review appears in both train and test.
6. Punctuation `. ! ? ,` is kept as tokens: `. ! ?` mark sentence boundaries (needed for `<s>` / `</s>` padding) and carry sentiment (`!`).

### A4. Build the balanced train & test subset

Exactly 2,500 + 2,500 train reviews  and 500 + 500 test reviews. Duplicates are removed first, and any test review that also appears in the train pool is dropped, so there is no leakage. The data is stored sorted by label, so we shuffle before sampling.

In [5]:
def dedupe(examples, exclude=frozenset()):
    seen, unique = set(exclude), []
    for text, label in examples:
        if text not in seen:
            seen.add(text)
            unique.append((text, label))
    return unique

def sample_balanced(examples, n_per_class, rng):
    pos = [ex for ex in examples if ex[1] == 1]
    neg = [ex for ex in examples if ex[1] == 0]
    rng.shuffle(pos)
    rng.shuffle(neg)
    chosen = pos[:n_per_class] + neg[:n_per_class]
    rng.shuffle(chosen)
    return chosen

def build_balanced_split(train_pool, test_pool, n_train_per_class=2500, n_test_per_class=500, seed=SEED):
    rng = random.Random(seed)
    train_unique = dedupe(train_pool)
    test_unique = dedupe(test_pool, exclude={t for t, _ in train_unique})
    print(f"Removed {len(train_pool) - len(train_unique)} duplicates from train pool, "
          f"{len(test_pool) - len(test_unique)} from test pool")
    return (sample_balanced(train_unique, n_train_per_class, rng),
            sample_balanced(test_unique, n_test_per_class, rng))

train_examples, test_examples = build_balanced_split(raw_train_examples, raw_test_examples)

def validate_examples(examples, name):
    assert examples, f"{name} is empty"
    for i, (text, label) in enumerate(examples):
        assert isinstance(text, str) and text.strip(), (name, i, "empty text")
        assert label in {0, 1}, (name, i, f"invalid label {label}")

validate_examples(train_examples, "train")
validate_examples(test_examples, "test")

train_counts = Counter(label for _, label in train_examples)
test_counts = Counter(label for _, label in test_examples)
print("Train size:", len(train_examples), dict(train_counts))
print("Test size: ", len(test_examples), dict(test_counts))

assert len(train_examples) == 5000 and train_counts == {1: 2500, 0: 2500}
assert len(test_examples) == 1000 and test_counts == {1: 500, 0: 500}
assert not ({t for t, _ in train_examples} & {t for t, _ in test_examples}), "train/test overlap"
print("Split checks passed (sizes, balance, no overlap).")

Removed 96 duplicates from train pool, 322 from test pool
Train size: 5000 {0: 2500, 1: 2500}
Test size:  1000 {1: 500, 0: 500}
Split checks passed (sizes, balance, no overlap).


In [6]:
def accuracy(labels, predictions):
    return float(np.mean(np.asarray(labels) == np.asarray(predictions)))

def confusion_counts(labels, predictions):
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    tp = int(np.sum((labels == 1) & (predictions == 1)))
    fp = int(np.sum((labels == 0) & (predictions == 1)))
    fn = int(np.sum((labels == 1) & (predictions == 0)))
    tn = int(np.sum((labels == 0) & (predictions == 0)))
    return tp, fp, fn, tn

def f1_score(labels, predictions, positive=1):
    """F1 of the positive class (2PR / (P+R)), 0.0 when undefined."""
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    if positive == 0:
        labels, predictions = 1 - labels, 1 - predictions
    tp, fp, fn, _ = confusion_counts(labels, predictions)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

y_train = np.array([label for _, label in train_examples])
y_test = np.array([label for _, label in test_examples])


# B. Pre-processing, tokenization, and vocabulary


Implements the decisions from A3.

In [7]:
HTML_TAG_RE = re.compile(r"<[^>]+>")
REPEAT_PUNCT_RE = re.compile(r"([.!?])[.!?]+")

def clean_text(text):
    text = HTML_TAG_RE.sub(" ", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.lower()
    text = REPEAT_PUNCT_RE.sub(r"\1", text)
    return text

example = "This movie was GREAT!!!<br /><br />I didn\u2019t expect it... really?!"
print("Raw:    ", example)
print("Cleaned:", clean_text(example))

Raw:     This movie was GREAT!!!<br /><br />I didn’t expect it... really?!
Cleaned: this movie was great!  i didn't expect it. really?


### B1. Data Pre-processing & Tokenization (task 1.1)

Tokens are words (with optional internal apostrophe) and the punctuation marks `. ! ? ,`. Anything else (quotes, brackets, hyphens, digits-only symbols) is dropped.

**Design choice:** a review is split into **sentences** at `. ! ?`, and **each sentence** is padded with `<s>` and `</s>`. This lets the bigram LM learn how sentences start and end, and later gives next-word generation a natural stopping point. A document is therefore a list of padded sentences.

In [8]:
BOS, EOS, UNK = "<s>", "</s>", "<unk>"
SPECIALS = [BOS, EOS, UNK]
SENT_END = {".", "!", "?"}
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?.,]")

def tokenize(clean):
    return TOKEN_RE.findall(clean)

def split_sentences(tokens):
    sentences, current = [], []
    for tok in tokens:
        current.append(tok)
        if tok in SENT_END:
            sentences.append(current)
            current = []
    if current:
        sentences.append(current)
    return sentences

def preprocess(text):
    """raw review -> list of sentences, each padded as [<s>, w1, ..., wn, </s>]"""
    sentences = split_sentences(tokenize(clean_text(text)))
    return [[BOS] + s + [EOS] for s in sentences]

demo = preprocess("Not good... really?! I didn't like it, at all.")
for s in demo:
    print(s)

assert preprocess("GOOD!") == [[BOS, "good", "!", EOS]]
assert preprocess("a b. c") == [[BOS, "a", "b", ".", EOS], [BOS, "c", EOS]]
assert preprocess("<br />") == []

['<s>', 'not', 'good', '.', '</s>']
['<s>', 'really', '?', '</s>']
['<s>', 'i', "didn't", 'like', 'it', ',', 'at', 'all', '.', '</s>']


In [9]:
train_docs_raw = [preprocess(text) for text, _ in tqdm(train_examples, desc="train")]
test_docs_raw = [preprocess(text) for text, _ in tqdm(test_examples, desc="test")]


train:   0%|          | 0/5000 [00:00<?, ?it/s]

test:   0%|          | 0/1000 [00:00<?, ?it/s]


- One **shared** vocabulary for both class LMs, so both use the same `V` in the Add-1 denominator and their perplexities are comparable.
- Words seen fewer than `MIN_COUNT` times in the **whole training set** become `<unk>`. This gives `<unk>` real training counts and lets the model score unseen test words instead of failing on them.
- The test set never creates new vocabulary entries.

In [10]:
MIN_COUNT = 2

def build_lm_vocab(docs, min_count=MIN_COUNT):
    counts = Counter(tok for doc in docs for sent in doc for tok in sent)
    kept = sorted(t for t, c in counts.items() if c >= min_count and t not in SPECIALS)
    return SPECIALS + kept

def replace_oov(docs, vocab_set):
    return [[[t if t in vocab_set else UNK for t in sent] for sent in doc] for doc in docs]

itos = build_lm_vocab(train_docs_raw)
vocab_set = set(itos)
V = len(itos)

train_docs = replace_oov(train_docs_raw, vocab_set)
test_docs = replace_oov(test_docs_raw, vocab_set)


print("Vocabulary size V:", V)
print("First 10 entries:", itos[:10])

Vocabulary size V: 24126
First 10 entries: ['<s>', '</s>', '<unk>', '!', ',', '.', '0', '00', '000', '00s']


# B2. Class-Conditional Language Models (task 1.2)

Bigrams are taken **within** each padded sentence, so no bigram ever crosses a sentence boundary (`</s>` is never a context and `<s>` is never a prediction target).

In [11]:
def extract_ngrams(tokens, n=2):
    return list(zip(*(tokens[i:] for i in range(n))))

print(extract_ngrams([BOS, "not", "good", ".", EOS], 2))

assert extract_ngrams([BOS, "a", EOS], 2) == [(BOS, "a"), ("a", EOS)]
assert extract_ngrams(["a"], 2) == []

[('<s>', 'not'), ('not', 'good'), ('good', '.'), ('.', '</s>')]


 Divide the training data by class

In [12]:

pos_docs = [doc for doc, y in zip(train_docs, y_train) if y == 1]
neg_docs = [doc for doc, y in zip(train_docs, y_train) if y == 0]

print("Positive training docs:", len(pos_docs))
print("Negative training docs:", len(neg_docs))
assert len(pos_docs) == 2500 and len(neg_docs) == 2500

Positive training docs: 2500
Negative training docs: 2500


 Implement the bigram LM with Add-1 smoothing

$$P(w_i \mid w_{i-1}) = \frac{c(w_{i-1}, w_i) + 1}{c(w_{i-1}) + V}$$

where $c(w_{i-1})$ is the number of bigrams that **start** with $w_{i-1}$ (so the distribution over the next word sums to exactly 1) and $V$ is the shared vocabulary size.

Counts are stored sparsely in dictionaries because a dense $V \times V$ matrix would be far too large. Unseen bigrams get the smoothed value $1 / (c(w_{i-1}) + V)$.

In [13]:
class BigramLM:
    def __init__(self, vocab_size):
        self.V = vocab_size
        self.bigram_counts = Counter()
        self.context_counts = Counter()
        self._log_denom = {}
        self._log_unseen_ctx = math.log(self.V)   # context never seen: (0 + V)

    def fit(self, docs):
        for doc in docs:
            for sent in doc:
                self.bigram_counts.update(extract_ngrams(sent, 2))
        for (w1, _), c in self.bigram_counts.items():
            self.context_counts[w1] += c
        self._log_denom = {w1: math.log(c + self.V) for w1, c in self.context_counts.items()}
        return self

    def log_prob(self, w1, w2):
        num = math.log(self.bigram_counts.get((w1, w2), 0) + 1)
        den = self._log_denom.get(w1, self._log_unseen_ctx)
        return num - den

    def prob(self, w1, w2):
        return math.exp(self.log_prob(w1, w2))

pos_lm = BigramLM(V).fit(pos_docs)
neg_lm = BigramLM(V).fit(neg_docs)

for name, lm in [("Positive LM", pos_lm), ("Negative LM", neg_lm)]:
    print(f"{name}: {len(lm.bigram_counts):,} distinct bigrams, "
          f"{sum(lm.bigram_counts.values()):,} bigram tokens, {len(lm.context_counts):,} contexts")

Positive LM: 230,218 distinct bigrams, 687,555 bigram tokens, 21,130 contexts
Negative LM: 218,686 distinct bigrams, 671,106 bigram tokens, 20,202 contexts


# B3. Log-likelihood and perplexity of a document (task 1.3)

For a document with $N$ predicted tokens (every bigram, i.e. all words plus `</s>` per sentence):

$$\log P(\text{doc}) = \sum_{\text{bigrams}} \log P(w_i \mid w_{i-1}) \qquad \text{PP}(\text{doc}) = \exp\!\left(-\frac{1}{N}\log P(\text{doc})\right)$$

Log-space avoids numerical underflow. Perplexity normalises by length, so reviews of different lengths are comparable.

In [14]:
def log_likelihood(doc, lm):
    """Return (total log-likelihood, number of predicted tokens N)."""
    total, n = 0.0, 0
    for sent in doc:
        for w1, w2 in zip(sent[:-1], sent[1:]):
            total += lm.log_prob(w1, w2)
            n += 1
    return total, n

def perplexity(doc, lm):
    ll, n = log_likelihood(doc, lm)
    return math.exp(-ll / n)

def score_document(doc, lm):
    ll, n = log_likelihood(doc, lm)
    return ll, math.exp(-ll / n)

doc = test_docs[0]
print("First test review (gold label =", y_test[0], ")")
for name, lm in [("Positive LM", pos_lm), ("Negative LM", neg_lm)]:
    ll, pp = score_document(doc, lm)
    print(f"  {name}: log-likelihood = {ll:9.2f}   perplexity = {pp:8.2f}")

First test review (gold label = 1 )
  Positive LM: log-likelihood =  -1675.08   perplexity =  1139.27
  Negative LM: log-likelihood =  -1692.39   perplexity =  1225.21
{'positive': {'log_likelihood': -48.65496959859136, 'perplexity': 129.73539587820596}, 'negative': {'log_likelihood': -53.26085657047117, 'perplexity': 205.63148459508525}}
